# Canonical Graph Recovery Slice

Purpose: prove the first canonical-graph recovery slice end to end, including explicit promotion, durable promoted assertion rows, and candidate-backed governance context.

Mode: proof

Related docs:
- `docs/plans/0001_successor_roadmap.md`
- `docs/plans/0006_phase11_graph_promotion_shape.md`
- `docs/adr/0012-start-canonical-graph-recovery-with-explicit-promotion-from-accepted-candidates.md`

## Phase 1

Purpose: create one accepted candidate with proposal, overlay, artifact, and epistemic context.

Input -> output: `review_seed_inputs -> accepted_candidate_with_context`

Acceptance criteria:
1. The candidate is accepted explicitly.
2. Mixed-mode proposal review and overlay application are explicit.
3. Artifact lineage and epistemic state are attached before promotion.

status: `proven`

execution_mode: `live`

In [1]:
from __future__ import annotations

import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "src").exists() and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent

if not (PROJECT_ROOT / "src").exists():
    raise RuntimeError("Could not locate onto-canon6 repo root from notebook cwd")

for candidate_path in (PROJECT_ROOT / "src", PROJECT_ROOT.parent / "llm_client"):
    if candidate_path.exists():
        candidate_str = str(candidate_path)
        if candidate_str not in sys.path:
            sys.path.insert(0, candidate_str)

import contextlib
import io
import json
from tempfile import TemporaryDirectory

from onto_canon6 import cli as cli_module
from onto_canon6.artifacts import ArtifactLineageService
from onto_canon6.extensions.epistemic import EpistemicService
from onto_canon6.pipeline import OverlayApplicationService, ReviewService


In [2]:
workspace = TemporaryDirectory()
workspace_path = Path(workspace.name)
review_db_path = workspace_path / 'review.sqlite3'
overlay_root = workspace_path / 'ontology_overlays'

def run_cli(args: list[str]) -> tuple[int, str, str]:
    stdout_buffer = io.StringIO()
    stderr_buffer = io.StringIO()
    with contextlib.redirect_stdout(stdout_buffer), contextlib.redirect_stderr(stderr_buffer):
        exit_code = cli_module.main(args)
    return exit_code, stdout_buffer.getvalue(), stderr_buffer.getvalue()

review_service = ReviewService(
    db_path=review_db_path,
    overlay_root=overlay_root,
    default_acceptance_policy='record_only',
)
overlay_service = OverlayApplicationService(
    db_path=review_db_path,
    overlay_root=overlay_root,
)
submission = review_service.submit_candidate_assertion(
    payload={
        'predicate': 'oc:hold_command_role_variant',
        'roles': {
            'commander': [
                {'entity_id': 'ent:person:eric_olson', 'entity_type': 'oc:person'}
            ],
            'organization': [
                {'entity_id': 'ent:org:ussocom', 'entity_type': 'oc:organization'}
            ],
            'title': [
                {'kind': 'value', 'value_kind': 'string', 'value': 'Commander'}
            ],
        },
    },
    profile_id='psyop_seed',
    profile_version='0.1.0',
    submitted_by='analyst:notebook',
    source_kind='text_file',
    source_ref='notes/promoted.txt',
    source_text='Eric Olson served as commander of USSOCOM.',
    claim_text='Eric Olson held the commander role at USSOCOM.',
    evidence_spans=({'start_char': 0, 'end_char': 10, 'text': 'Eric Olson'},),
)
proposal = review_service.review_proposal(
    proposal_id=submission.proposals[0].proposal_id,
    decision='accepted',
    actor_id='analyst:reviewer',
    acceptance_policy='apply_to_overlay',
)
overlay_application = overlay_service.apply_proposal_to_overlay(
    proposal_id=proposal.proposal_id,
    applied_by='analyst:overlay',
)
accepted_candidate = review_service.review_candidate(
    candidate_id=submission.candidate.candidate_id,
    decision='accepted',
    actor_id='analyst:reviewer',
)
artifact_service = ArtifactLineageService(db_path=review_db_path)
artifact = artifact_service.register_artifact(
    artifact_kind='source',
    uri='notes/promoted.txt',
    label='promoted note',
)
artifact_service.link_candidate_artifact(
    candidate_id=accepted_candidate.candidate_id,
    artifact_id=artifact.artifact_id,
    support_kind='quoted_from',
    reference_detail='primary mention',
)
EpistemicService(db_path=review_db_path).record_confidence(
    candidate_id=accepted_candidate.candidate_id,
    confidence_score=0.91,
    source_kind='user',
    actor_id='analyst:confidence',
    rationale='Accepted before promotion.',
)
seed_summary = {
    'candidate_id': accepted_candidate.candidate_id,
    'proposal_id': proposal.proposal_id,
    'overlay_application_id': overlay_application.application_id,
    'artifact_id': artifact.artifact_id,
}
seed_summary

{'candidate_id': 'cand_719f3a5a662d43ada149c499',
 'proposal_id': 'prop_43bccc1fb0f31775139f4503',
 'overlay_application_id': 'oapp_c145f6971ada43c38007005f',
 'artifact_id': 'art_858f2d9dea7542fd86a550ce'}

## Phase 2

Purpose: promote the accepted candidate and inspect the durable promoted assertion rows through the CLI.

Input -> output: `accepted_candidate_with_context -> promoted_assertion_rows`

Acceptance criteria:
1. Promotion is explicit and auditable.
2. Recovered graph rows are inspectable without direct SQL.
3. The promoted assertion preserves its source candidate link.

status: `proven`

execution_mode: `live`

In [3]:
promote_exit, promote_stdout, promote_stderr = run_cli([
    'promote-candidate',
    '--candidate-id', accepted_candidate.candidate_id,
    '--actor-id', 'analyst:graph-promoter',
    '--review-db-path', str(review_db_path),
    '--output', 'json',
])
assert promote_exit == 0, promote_stderr
promotion_result = json.loads(promote_stdout)
assert promotion_result['assertion']['source_candidate_id'] == accepted_candidate.candidate_id
assert len(promotion_result['entities']) == 2

list_exit, list_stdout, list_stderr = run_cli([
    'list-promoted-assertions',
    '--review-db-path', str(review_db_path),
    '--output', 'json',
])
assert list_exit == 0, list_stderr
promoted_assertions = json.loads(list_stdout)
assert len(promoted_assertions) == 1
promoted_assertions

[{'assertion_id': 'gassert_12e1bf7aa6792cc10d8f488d',
  'claim_text': 'Eric Olson held the commander role at USSOCOM.',
  'normalized_body': {'predicate': 'oc:hold_command_role_variant',
   'roles': {'commander': [{'entity_id': 'ent:person:eric_olson',
      'entity_type': 'oc:person',
      'kind': 'entity'}],
    'organization': [{'entity_id': 'ent:org:ussocom',
      'entity_type': 'oc:organization',
      'kind': 'entity'}],
    'title': [{'kind': 'value',
      'value': 'Commander',
      'value_kind': 'string'}]}},
  'predicate': 'oc:hold_command_role_variant',
  'profile': {'profile_id': 'psyop_seed', 'profile_version': '0.1.0'},
  'promoted_at': '2026-03-18T19:23:03.124787+00:00',
  'promoted_by': 'analyst:graph-promoter',
  'source_candidate_id': 'cand_719f3a5a662d43ada149c499'}]

## Phase 3

Purpose: export the promoted graph report that traverses candidate-backed governance, artifact, and epistemic context.

Input -> output: `promoted_assertion_rows -> promoted_graph_report`

Acceptance criteria:
1. The report includes the promoted assertion and its source candidate.
2. Linked proposal, overlay, artifact, and confidence state are visible without duplication.
3. The output remains a downstream JSON artifact.

status: `proven`

execution_mode: `live`

In [4]:
report_exit, report_stdout, report_stderr = run_cli([
    'export-promoted-graph-report',
    '--review-db-path', str(review_db_path),
    '--output', 'json',
])
assert report_exit == 0, report_stderr
promoted_graph_report = json.loads(report_stdout)
assert promoted_graph_report['summary']['total_assertions'] == 1
assert promoted_graph_report['summary']['total_entities'] == 2
assert promoted_graph_report['summary']['total_assertions_with_artifacts'] == 1
assert promoted_graph_report['summary']['total_assertions_with_confidence'] == 1
assert promoted_graph_report['assertion_bundles'][0]['source_candidate']['candidate_id'] == accepted_candidate.candidate_id
assert promoted_graph_report['assertion_bundles'][0]['linked_proposals'][0]['proposal_id'] == proposal.proposal_id
assert promoted_graph_report['assertion_bundles'][0]['artifact_links'][0]['artifact_id'] == artifact.artifact_id
promoted_graph_report

{'assertion_bundles': [{'artifact_links': [{'artifact_id': 'art_858f2d9dea7542fd86a550ce',
     'candidate_id': 'cand_719f3a5a662d43ada149c499',
     'created_at': '2026-03-18T19:23:03.070158+00:00',
     'reference_detail': 'primary mention',
     'support_kind': 'quoted_from'}],
   'artifacts': [{'artifact_id': 'art_858f2d9dea7542fd86a550ce',
     'artifact_kind': 'source',
     'created_at': '2026-03-18T19:23:03.066392+00:00',
     'fingerprint': None,
     'label': 'promoted note',
     'metadata': {},
     'uri': 'notes/promoted.txt'}],
   'assertion': {'assertion_id': 'gassert_12e1bf7aa6792cc10d8f488d',
    'claim_text': 'Eric Olson held the commander role at USSOCOM.',
    'normalized_body': {'predicate': 'oc:hold_command_role_variant',
     'roles': {'commander': [{'entity_id': 'ent:person:eric_olson',
        'entity_type': 'oc:person',
        'kind': 'entity'}],
      'organization': [{'entity_id': 'ent:org:ussocom',
        'entity_type': 'oc:organization',
        'kind': 

In [5]:
workspace.cleanup()
'cleanup_complete'

'cleanup_complete'